# SpatialArtifacts: Standard Visium Tutorial

This notebook demonstrates the `SpatialArtifacts` workflow for detecting and classifying edge artifacts in 10x Genomics Visium spatial transcriptomics data.

The workflow has two steps:
1. **`detect_edge_artifacts()`** — identifies outlier spots using MAD-based detection combined with morphological image processing
2. **`classify_edge_artifacts()`** — assigns hierarchical labels based on location (edge vs interior) and size (large vs small)

**Data:** Replace `SAMPLE_PATH` and `SAMPLE_ID` with your own Space Ranger output path.

**Reference:** He, Thompson, Totty, and Hicks (in preparation), SpatialArtifacts package.

## Installation

```bash
pip install spatial-artifacts
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

from spatial_artifacts import detect_edge_artifacts, classify_edge_artifacts

## Parameters

Set your sample path and ID here.

In [ ]:
SAMPLE_PATH = Path("/path/to/your/spaceranger/outs/")
SAMPLE_ID = "your_sample_id"
MAD_THRESHOLD = 2

## Load Data

In [ ]:
adata = sc.read_visium(SAMPLE_PATH)
adata.var_names_make_unique()
print(f"Loaded: {adata.n_obs} spots, {adata.n_vars} genes")

## QC Metrics and Spatial Coordinates

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.obs["sum"] = adata.obs["total_counts"]
adata.obs["sample_id"] = SAMPLE_ID
adata.obs["in_tissue"] = adata.obs["in_tissue"].astype(bool)
adata = adata[adata.obs["in_tissue"]].copy()
print(f"In-tissue spots: {adata.n_obs}")

adata.obs["pxl_col_in_fullres"] = adata.obsm["spatial"][:, 0]
adata.obs["pxl_row_in_fullres"] = adata.obsm["spatial"][:, 1]

spatial_dir = SAMPLE_PATH / "spatial"
tissue_pos = pd.read_csv(
    spatial_dir / "tissue_positions_list.csv",
    header=None,
    names=["barcode", "in_tissue", "array_row", "array_col",
           "pxl_row_in_fullres", "pxl_col_in_fullres"]
).set_index("barcode")

adata.obs["array_row"] = tissue_pos.loc[adata.obs_names, "array_row"].values
adata.obs["array_col"] = tissue_pos.loc[adata.obs_names, "array_col"].values

print(f"array_row range: {adata.obs['array_row'].min()} - {adata.obs['array_row'].max()}")
print(f"array_col range: {adata.obs['array_col'].min()} - {adata.obs['array_col'].max()}")

## Step 1: Detect Edge Artifacts

`detect_edge_artifacts()` combines MAD-based outlier detection with morphological image processing (fill, outline, and star patterns) to identify clusters of low-quality spots, then evaluates whether those clusters touch tissue boundaries.

Key parameters:
- `mad_threshold`: sensitivity for outlier detection (lower = more sensitive)
- `edge_threshold`: minimum boundary coverage to call a cluster an edge artifact
- `min_cluster_size`: small isolated normal regions below this size are filled in during morphological cleaning

In [ ]:
adata = detect_edge_artifacts(
    adata,
    platform="visium",
    qc_metric="total_counts",
    samples="sample_id",
    mad_threshold=MAD_THRESHOLD,
    edge_threshold=0.5,
    min_cluster_size=40,
    batch_var="sample_id",
    name="edge_artifact",
    verbose=True,
)

print("\nRaw edge detection:")
print(adata.obs["edge_artifact_edge"].value_counts())

## Step 2: Classify Edge Artifacts

`classify_edge_artifacts()` applies a 2x2 hierarchical logic based on:
- **Location**: edge (`_edge = True`) vs interior
- **Size**: large (`> min_spots`) vs small

This produces five categories:
- `not_artifact`
- `large_edge_artifact`
- `small_edge_artifact`
- `large_interior_artifact`
- `small_interior_artifact`

In [ ]:
adata = classify_edge_artifacts(
    adata,
    min_spots=20,
    name="edge_artifact",
    verbose=True,
)

print("\nClassification summary:")
print(adata.obs["edge_artifact_classification"].value_counts())

## Visualization

Three panels showing:
- **A**: Total UMI counts (log10 scale)
- **B**: Standard global QC threshold (< 1000 UMI) for comparison
- **C**: SpatialArtifacts hierarchical classification

In [ ]:
artifact_colors = {
    "not_artifact":            "lightgray",
    "small_edge_artifact":     "#FFB347",
    "large_edge_artifact":     "#FF4500",
    "small_interior_artifact": "#00CED1",
    "large_interior_artifact": "#0047AB",
}

obs = adata.obs.copy()
x = obs["pxl_col_in_fullres"].values
y = -obs["pxl_row_in_fullres"].values

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Total UMI (log10)
log_umi = np.log10(obs["sum"].values + 1)
sc_a = axes[0].scatter(x, y, c=log_umi, cmap="magma", s=4, alpha=0.9)
plt.colorbar(sc_a, ax=axes[0], label="log10(UMI)")
axes[0].set_title("A. Total UMI Distribution", fontweight="bold")
axes[0].axis("off")
axes[0].set_aspect("equal")

# Panel B: Standard QC threshold (< 1000 UMI)
std_threshold = 1000
std_colors = np.where(obs["sum"].values < std_threshold, "black", "#F0F0F0")
axes[1].scatter(x, y, c=std_colors, s=4, alpha=0.9)
axes[1].set_title(f"B. Standard QC (< {std_threshold} UMI)", fontweight="bold")
axes[1].axis("off")
axes[1].set_aspect("equal")

# Panel C: SpatialArtifacts classification
cls = obs["edge_artifact_classification"].values
order = np.argsort(cls == "not_artifact")[::-1]
colors_c = [artifact_colors.get(c, "lightgray") for c in cls[order]]
axes[2].scatter(x[order], y[order], c=colors_c, s=4, alpha=0.9)
axes[2].set_title("C. SpatialArtifacts Classification", fontweight="bold")
axes[2].axis("off")
axes[2].set_aspect("equal")

for label, color in artifact_colors.items():
    if label in cls:
        axes[2].scatter([], [], c=color, label=label.replace("_", " "), s=20)
axes[2].legend(loc="lower right", fontsize=7, framealpha=0.8)

plt.suptitle(f"Sample: {SAMPLE_ID}", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Filtering (Optional)

Remove edge artifact spots for downstream analysis. Here we remove both large and small edge artifacts while keeping interior artifacts for further review.

In [ ]:
spots_to_keep = ~adata.obs["edge_artifact_classification"].isin(
    ["large_edge_artifact", "small_edge_artifact"]
)
adata_filtered = adata[spots_to_keep].copy()

print(f"Original spots:  {adata.n_obs}")
print(f"After filtering: {adata_filtered.n_obs}")
print(f"Removed:         {adata.n_obs - adata_filtered.n_obs} spots")

## Session Info

In [ ]:
sc.logging.print_versions()